<a href="https://colab.research.google.com/github/anastasia-kotselivska/ITA/blob/main/lab5/Lab5_NLP_Kotselivska.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch pandas numpy

In [3]:
# Читання ідеального розпізнаного тексту
with open('ocr_result.txt', 'r', encoding='utf-8') as f:
  clean_text = f.read()
# Читання тексту, розпізнаного з розмитого/повернутого фото
with open('ocr_result_degraded.txt', 'r', encoding='utf-8') as f:
  noisy_text = f.read()
print("--- Clean Text ---")
print(clean_text[:200])

--- Clean Text ---
М ІВАНО-ФРАНКІВСЬК
ВУЛ: КОБИЛЯНСЬКОі,56
ПН 325497319188
Кaca 09
ВІдділ Супернаркет
Касир 160/12670
К9 ПК121 14/2/
2263088
Штрих код 4820242913786
1049932 Eskaro Стрічка малярна дел
фІол
351.14
Ж 129.0


In [4]:
import re
# 1. Базові шаблони (з методички) для порівняння на чистому та зашумленому тексті
date_pattern = r"\b\d{2}[-./]\d{2}[-./]\d{2,4}\b"
basic_amount_pattern = r"(?i)(?:сума|разом|total)[\s:.-]*(\d+[.,]\d{2})"
# 2. Цільові шаблони під Варіант 6:
# - Проміжна сума / знижка (SUBTOTAL, Знижка/Знихка)
# - Підсумкова сума (TOTAL, Сума, а також число після "Сума" навіть через перенос рядка/пробіли)
subtotal_pattern = (
    r"(?i)\b(?:sub(?:total)?|зн[иі][жх]к[ая])\b"
    r"[\s:.,-]*(\d{1,6}(?:[\s.,]\d{2})?)"
)
total_pattern = (
    r"(?i)(?:\btot[a1l]l\b|\bсу[мн][ая]\b)"
    r"[\s:.,-]*(\d{1,6}(?:[\s.,]\d{2})?)"
)
print("[*] Пошук базових RegEx у ЧИСТОМУ тексті:")
print("Дата:", re.findall(date_pattern, clean_text))
print("Сума (базовий):", re.findall(basic_amount_pattern, clean_text))

print("\n[*] Пошук базових RegEx у ЗАШУМЛЕНОМУ тексті:")
print("Дата:", re.findall(date_pattern, noisy_text))
print("Сума (базовий):", re.findall(basic_amount_pattern, noisy_text))

print("\n[+] Пошук за Варіантом 6 у ЧИСТОМУ тексті:")
print("Проміжна / знижка (SUBTOTAL / Знижка):", re.findall(subtotal_pattern, clean_text))
print("Підсумкова сума (TOTAL / Сума):", re.findall(total_pattern, clean_text))

print("\n[+] Пошук за Варіантом 6 у ЗАШУМЛЕНОМУ тексті:")
print("Проміжна / знижка (SUBTOTAL / Знижка):", re.findall(subtotal_pattern, noisy_text))
print("Підсумкова сума (TOTAL / Сума):", re.findall(total_pattern, noisy_text))

[*] Пошук базових RegEx у ЧИСТОМУ тексті:
Дата: ['13.10.2026']
Сума (базовий): ['1933.21']

[*] Пошук базових RegEx у ЗАШУМЛЕНОМУ тексті:
Дата: []
Сума (базовий): []

[+] Пошук за Варіантом 6 у ЧИСТОМУ тексті:
Проміжна / знижка (SUBTOTAL / Знижка): ['59.79']
Підсумкова сума (TOTAL / Сума): ['1933.21']

[+] Пошук за Варіантом 6 у ЗАШУМЛЕНОМУ тексті:
Проміжна / знижка (SUBTOTAL / Знижка): []
Підсумкова сума (TOTAL / Сума): ['1933']


In [10]:
from transformers import pipeline

# Попередня нормалізація тексту з урахуванням специфіки чека (укр/англ + OCR-помилки)
def preprocess_ocr_text(text: str) -> str:
    # Усунення зайвих пробілів та переносів рядків
    cleaned = " ".join(text.split())

    # Виправлення типових символьних підмін у чистому та деградованому чеках
    replacements = {
        "ІRAHП-ФРАНКІВСЬК": "Івано-Франківськ",
        "ІВАІО-ФРАНКІВСЬКА": "Івано-Франківська",
        "RYЛ , КОБИЛЯНСЬКІі 56": "вул: Кобилянської, 56",
        "КОБИЛЯНСЬКОі,56": "Кобилянської, 56",
        "ПРМВАТБАНК": "ПриватБанк",
        "Супернаркет": "Супермаркет",
        "Гернетик": "Герметик",
        "Знихка": "Знижка",
        "Сумя": "Сума",
        "ОПЛНТА": "ОПЛАТА",
        "ФІскельнии": "Фіскальний",
        "Kirch |and PrifeTex": "Kirchland PrimeTex",
        "GRIVER": "GRUVER",
    }
    for bad, good in replacements.items():
        cleaned = cleaned.replace(bad, good)

    # Прибираємо зайві пробіли перед розділовими знаками
    cleaned = re.sub(r"\s+([,.:;!?])", r"\1", cleaned)
    return cleaned

clean_text_norm = preprocess_ocr_text(clean_text)
noisy_text_norm = preprocess_ocr_text(noisy_text)

print("[*] Завантаження моделі NER...")
ner_pipeline = pipeline(
    "ner",
    model="Babelscape/wikineural-multilingual-ner",
    aggregation_strategy="simple",  # Об'єднує B-XXX та I-XXX у єдину сутність
)
print("[+] Модель успішно завантажена!")

print("\n--- Нормалізований Clean Text (фрагмент) ---")
print(clean_text_norm[:250])
print("\n--- Нормалізований Noisy Text (фрагмент) ---")
print(noisy_text_norm[:250])

[*] Завантаження моделі NER...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[+] Модель успішно завантажена!

--- Нормалізований Clean Text (фрагмент) ---
М ІВАНО-ФРАНКІВСЬК ВУЛ: Кобилянської, 56 ПН 325497319188 Кaca 09 ВІдділ Супермаркет Касир 160/12670 К9 ПК121 14/2/ 2263088 Штрих код 4820242913786 1049932 Eskaro Стрічка малярна дел фІол 351.14 Ж 129.01 Штрих код 5902120208609 654422 GRUVER Герметик 

--- Нормалізований Noisy Text (фрагмент) ---
Івано-Франківськ вул: Кобилянської, 56 ПIН 325497419188 Kara 09 КLДДАЛ Штрих код 4820242913786 1049932 Eskaro Стрічка налнона ДЕП ОЛ ЕКСПЕРТ 48нН 351.14 X 129,01 654422 GRUVER Герне ИК П;М # 146 ШТрнх ход 5902121215717 ':;к; ';" ШАН 589822 Kirchland 


In [11]:
print("[*] Аналіз чистого тексту через NER...")
entities_clean = ner_pipeline(clean_text_norm)

print("\n[+] Знайдені сутності (Clean Text):")
for entity in entities_clean:
    print(
        f"[{entity['entity_group']}] (Score: {entity['score']:.2f}) -> {entity['word']}"
        f" (chars {entity['start']}:{entity['end']})"
    )

print("\n[*] Аналіз зашумленого тексту через NER...")
entities_noisy = ner_pipeline(noisy_text_norm)

print("\n[+] Знайдені сутності (Noisy Text):")
for entity in entities_noisy:
    print(
        f"[{entity['entity_group']}] (Score: {entity['score']:.2f}) -> {entity['word']}"
        f" (chars {entity['start']}:{entity['end']})"
    )

# Дослідження пайплайну без агрегації (aggregation_strategy=None)
ner_raw = pipeline(
    "ner",
    model="Babelscape/wikineural-multilingual-ner",
    aggregation_strategy=None,
)

raw_tokens_clean = ner_raw(clean_text_norm)

print("\n[*] Сирі токени, сабтокени '##' та BIO-мітки (Clean Text):")
for tok in raw_tokens_clean[:40]:
    print(
        f"  token={tok['word']:20s} | tag={tok['entity']:10s} | score={tok['score']:.4f}"
    )

# Окремо покажемо сабтокени з префіксом '##' (якщо вони є у виводі)
subwords = [t for t in raw_tokens_clean if t["word"].startswith("##")]
print(f"\n[+] Кількість сабтокенів із '##' у перших токенах: {len(subwords)}")
for sw in subwords[:15]:
    print(
        f"  subword={sw['word']:18s} | tag={sw['entity']:10s} | score={sw['score']:.4f}"
    )

[*] Аналіз чистого тексту через NER...

[+] Знайдені сутності (Clean Text):
[LOC] (Score: 0.67) -> Кобилян (chars 24:31)
[MISC] (Score: 0.49) -> Супермар (chars 72:80)
[MISC] (Score: 0.45) -> Касир (chars 84:89)
[MISC] (Score: 0.44) -> Eskar (chars 155:160)
[MISC] (Score: 0.69) -> Kirchland PrimeTex (chars 421:439)
[MISC] (Score: 0.38) -> Су (chars 674:676)

[*] Аналіз зашумленого тексту через NER...

[+] Знайдені сутності (Noisy Text):
[LOC] (Score: 0.99) -> Івано - Франківськ (chars 0:16)
[LOC] (Score: 0.88) -> Кобилянської (chars 22:34)
[MISC] (Score: 0.66) -> Kirchland PrimeTex (chars 240:258)
[MISC] (Score: 0.46) -> Гру (chars 263:266)
[ORG] (Score: 0.50) -> При (chars 302:305)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


[*] Сирі токени, сабтокени '##' та BIO-мітки (Clean Text):
  token=К                    | tag=B-LOC      | score=0.8277
  token=##оби                | tag=I-LOC      | score=0.6197
  token=##лян                | tag=I-LOC      | score=0.5727
  token=Су                   | tag=I-MISC     | score=0.4777
  token=##пер                | tag=I-MISC     | score=0.5458
  token=##мар                | tag=I-MISC     | score=0.4457
  token=К                    | tag=I-MISC     | score=0.4020
  token=##ас                 | tag=I-MISC     | score=0.4763
  token=##ир                 | tag=I-MISC     | score=0.4718
  token=Es                   | tag=I-MISC     | score=0.3952
  token=##kar                | tag=I-MISC     | score=0.4872
  token=Ki                   | tag=I-MISC     | score=0.6474
  token=##rch                | tag=I-MISC     | score=0.7710
  token=##land               | tag=I-MISC     | score=0.7257
  token=Prime                | tag=I-MISC     | score=0.7347
  token=##T              

In [12]:
import numpy as np
import pandas as pd

models_to_compare = [
    "dslim/bert-base-NER",
    "Babelscape/wikineural-multilingual-ner",
    "elastic/distilbert-base-cased-finetuned-conll03-english",
]

def normalize_entity_token(w: str) -> str:
    return re.sub(r"[^a-zA-Zа-яА-ЯіїєґІЇЄҐ0-9]", "", w.lower())

metrics_rows = []
model_outputs = {}

for m_name in models_to_compare:
    print(f"[*] Завантаження та запуск моделі: {m_name}...")
    nlp_pipe = pipeline("ner", model=m_name, aggregation_strategy="simple")

    ent_clean = nlp_pipe(clean_text_norm)
    ent_noisy = nlp_pipe(noisy_text_norm)
    model_outputs[m_name] = (ent_clean, ent_noisy)

    map_clean = {
        normalize_entity_token(e["word"]): e["entity_group"]
        for e in ent_clean
        if normalize_entity_token(e["word"])
    }
    map_noisy = {
        normalize_entity_token(e["word"]): e["entity_group"]
        for e in ent_noisy
        if normalize_entity_token(e["word"])
    }

    clean_keys = set(map_clean.keys())
    noisy_keys = set(map_noisy.keys())
    matched_keys = clean_keys.intersection(noisy_keys)

    # Preservation Rate (PR): частка сутностей із чистого тексту, що збереглися у зашумленому
    pr = (len(matched_keys) / len(clean_keys) * 100.0) if clean_keys else 0.0

    avg_score_clean = float(np.mean([e["score"] for e in ent_clean])) if ent_clean else 0.0
    avg_score_noisy = float(np.mean([e["score"] for e in ent_noisy])) if ent_noisy else 0.0
    delta_score = avg_score_clean - avg_score_noisy

    # Class Drift: скільки спільних сутностей змінили свій клас між чистим і зашумленим текстом
    class_drift = sum(1 for k in matched_keys if map_clean[k] != map_noisy[k])

    metrics_rows.append({
        "Model": m_name,
        "Clean Ent.": len(ent_clean),
        "Noisy Ent.": len(ent_noisy),
        "PR (%)": round(pr, 2),
        "Score Clean": round(avg_score_clean, 4),
        "Score Noisy": round(avg_score_noisy, 4),
        "Delta Score": round(delta_score, 4),
        "Class Drift": class_drift,
    })

df_metrics = pd.DataFrame(metrics_rows)
print("\n=== Порівняльна таблиця моделей NER (Кроки 5–6) ===")
print(df_metrics.to_string(index=False))

[*] Завантаження та запуск моделі: dslim/bert-base-NER...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[*] Завантаження та запуск моделі: Babelscape/wikineural-multilingual-ner...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[*] Завантаження та запуск моделі: elastic/distilbert-base-cased-finetuned-conll03-english...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


=== Порівняльна таблиця моделей NER (Кроки 5–6) ===
                                                  Model  Clean Ent.  Noisy Ent.  PR (%)  Score Clean  Score Noisy  Delta Score  Class Drift
                                    dslim/bert-base-NER          12          15   40.00       0.7731       0.7438       0.0292            0
                 Babelscape/wikineural-multilingual-ner           6           5   16.67       0.5219       0.6954      -0.1735            0
elastic/distilbert-base-cased-finetuned-conll03-english          23          37   66.67       0.4758       0.6048      -0.1290            5


In [13]:
import json

# Обираємо еталонну модель (наприклад, найвищий PR із урахуванням мінімального Class Drift,
# або явно фіксуємо Babelscape/wikineural-multilingual-ner для укр/англ чека)
best_model_name = "Babelscape/wikineural-multilingual-ner"
best_entities_clean, _ = model_outputs[best_model_name]

ner_export = {
    "variant": 6,
    "source_file": "ocr_result_clean.txt",
    "model_name": best_model_name,
    "aggregation_strategy": "simple",
    "entities_count": len(best_entities_clean),
    "entities": [
        {
            "entity_group": item["entity_group"],
            "word": item["word"],
            "score": round(float(item["score"]), 4),
            "start": int(item["start"]),
            "end": int(item["end"]),
        }
        for item in best_entities_clean
    ],
}

with open("ner_extracted.json", "w", encoding="utf-8") as f:
    json.dump(ner_export, f, ensure_ascii=False, indent=2)

print(f"[+] Еталонна модель: {best_model_name}")
print("[+] Файл ner_extracted.json успішно сформовано:")
print(json.dumps(ner_export, ensure_ascii=False, indent=2))

[+] Еталонна модель: Babelscape/wikineural-multilingual-ner
[+] Файл ner_extracted.json успішно сформовано:
{
  "variant": 6,
  "source_file": "ocr_result_clean.txt",
  "model_name": "Babelscape/wikineural-multilingual-ner",
  "aggregation_strategy": "simple",
  "entities_count": 6,
  "entities": [
    {
      "entity_group": "LOC",
      "word": "Кобилян",
      "score": 0.6734,
      "start": 24,
      "end": 31
    },
    {
      "entity_group": "MISC",
      "word": "Супермар",
      "score": 0.4897,
      "start": 72,
      "end": 80
    },
    {
      "entity_group": "MISC",
      "word": "Касир",
      "score": 0.4501,
      "start": 84,
      "end": 89
    },
    {
      "entity_group": "MISC",
      "word": "Eskar",
      "score": 0.4412,
      "start": 155,
      "end": 160
    },
    {
      "entity_group": "MISC",
      "word": "Kirchland PrimeTex",
      "score": 0.6933,
      "start": 421,
      "end": 439
    },
    {
      "entity_group": "MISC",
      "word": "Су",
   